<a href="https://colab.research.google.com/github/areejtechcampus/AI-Agent/blob/main/Project_Barcode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [38]:
!apt-get install -y libzbar0
!pip install pyzbar Pillow langgraph
!pip install -q -U langchain-openai
!pip install -q openpyxl


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libzbar0 is already the newest version (0.23.92-4build2).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [39]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [40]:
import os
from pyzbar.pyzbar import decode
from PIL import Image
from typing import TypedDict, List
folder_path = '/content/drive/MyDrive/barcodes'
class AgentState(TypedDict):
   folder_path: str #
   extracted_urls: List[str]
   job_details: List[dict]
   final_report: str #




In [41]:
def extract_barcodes_node(state: AgentState):
    print("--- Exreracting links from barcodes---")
    urls = []
    folder = state['folder_path']

    for filename in os.listdir(folder):
        if filename.endswith((".png", ".jpg", ".jpeg")):
            path = os.path.join(folder, filename)

            decoded = decode(Image.open(path))
            for obj in decoded:
                url = obj.data.decode('utf-8')
                urls.append(url)


    return {"extracted_urls": list(set(urls))} #

In [42]:
import requests
from bs4 import BeautifulSoup



def scrape_jobs_node(state: AgentState):
    print(f"--- جاري تحليل {len(state['extracted_urls'])} روابط ---")
    jobs_info = []

    for url in state['extracted_urls']:
        try:

            response = requests.get(url, timeout=10)
            soup = BeautifulSoup(response.text, 'html.parser')
            text = soup.get_text(separator=' ', strip=True)[:2000]
            jobs_info.append({"url": url, "content": text})
        except Exception as e:

            print(f"فشل الوصول للرابط: {url} بسبب: {e}")

    return {"job_details": jobs_info}

In [43]:
import os
import time
from google.colab import userdata
from langchain_openai import ChatOpenAI

def match_evaluator_node(state: AgentState):
    print("--- جاري تقييم الوظائف باستخدام DeepSeek (مجاناً ومستقر) ---")

    # 1. جلب المفتاح من Secrets
    openrouter_key = userdata.get('OpenRouter')

    # 2. إعداد الموديل (تأكدي أن الإزاحة هنا صحيحة داخل الدالة)
    # تم تغيير الموديل من deepseek/deepseek-chat:free إلى openai/gpt-3.5-turbo
    # يمكنك تجربة موديلات أخرى متاحة على openrouter إذا واجهت مشكلة مع هذا الموديل
    llm = ChatOpenAI(
        model="openai/gpt-3.5-turbo",
        openai_api_key=openrouter_key,
        openai_api_base="https://openrouter.ai/api/v1",
        temperature=0
    )

    # بروفايل أريج المتميز
    my_profile = """
    الاسم: أريج الزايدي.
    المؤهل: خريجة تقنية معلومات من جامعة الطائف.
    الشهادات الاحترافية:
    - AWS Certified DevOps Engineer - Professional (تدريب 100 ساعة).
    - AWS Certified Cloud Practitioner.
    - eJPT (Junior Penetration Tester).
    - Security+.
    - COBIT 5 (GRC & Governance).
    الخبرات: التدريب في معسكر سدايا (SDA)، خبرة في سياسات NCA، وبرمجة Python و LangChain.
    الموقع المفضل: جدة أو مكة.
    """

    results = []

    # 3. حلقة التكرار لمعالجة الـ 33 رابطاً
    for job in state['job_details']:
        try:
            # تقليص المحتوى لضمان عدم تجاوز حدود الـ Tokens المجانية
            job_content = job['content'][:2000]

            prompt = f"""
            قارن بين ملفي الشخصي:
            {my_profile}

            وبين وصف هذه الوظيفة:
            {job_content}

            بناءً على المقارنة، أعطني:
            1. درجة مطابقة من 10.
            2. ملخص سريع يوضح نقاط القوة ونقاط النقص بالنسبة لي لهذه الوظيفة.
            """

            # استدعاء الموديل
            response = llm.invoke(prompt)

            # إضافة النتيجة للقائمة
            results.append(f"الرابط: {job['url']}\nالتقييم:\n{response.content}\n")

            # انتظار بسيط لتجنب الـ Rate Limit
            time.sleep(1)

        except Exception as e:
            print(f"تخطيت رابط بسبب خطأ بسيط: {e}")
            continue

    # 4. إرجاع التقرير النهائي
    return {"final_report": "\n---\n".join(results)}

In [44]:
import pandas as pd

file_name = '/content/Job_Analysis_Report.xlsx'
df_report = pd.read_excel(file_name)
display(df_report)

""


In [45]:
import pandas as pd
from langgraph.graph import StateGraph, END

# --- إعداد المخطط ---
workflow = StateGraph(AgentState)
workflow.add_node("extractor", extract_barcodes_node)
workflow.add_node("scraper", scrape_jobs_node)
workflow.add_node("matcher", match_evaluator_node)

workflow.set_entry_point("extractor")
workflow.add_edge("extractor", "scraper")
workflow.add_edge("scraper", "matcher")
workflow.add_edge("matcher", END)

app = workflow.compile()

# --- التشغيل ---
inputs = {"folder_path": "/content/drive/MyDrive/barcodes"}

try:
    print("--- جاري تشغيل الـ Agent... قد يستغرق ذلك وقتاً حسب عدد الروابط ---")
    final_output = app.invoke(inputs)

    # 1. استخراج النص الخام (الذي يظهر في الشاشة)
    report_text = final_output.get('final_report', "")

    # 2. تحويل النتائج إلى Excel
    # ملاحظة: نفترض هنا أن التقرير يحتوي على روابط وتقييمات مفصولة بـ "---" أو مخزنة كقائمة
    # الأفضل هو تحويل القائمة مباشرة إذا كانت متوفرة في الـ state

    # سنقوم بتقسيم النص الناتج وتحويله لجدول بسيط
    data_rows = []
    sections = report_text.split("---") # تقسيم بناءً على الفاصل اللي استخدمناه في الكود

    for section in sections:
        if "الرابط:" in section:
            # تنظيف واستخراج البيانات (مثال بسيط)
            url = section.split("التقييم:")[0].replace("الرابط:", "").strip()
            evaluation = section.split("التقييم:")[1].strip() if "التقييم:" in section else ""
            data_rows.append({"Link": url, "AI_Evaluation": evaluation})

    # إنشاء DataFrame
    df = pd.DataFrame(data_rows)

    # 3. حفظ الملف بصيغة Excel
    file_name = "Job_Analysis_Report.xlsx"
    df.to_excel(file_name, index=False)

    print("\n" + "="*30)
    print(f"✅ تم استخراج التقرير بنجاح!")
    print(f"📁 اسم الملف: {file_name}")
    print("="*30)

    # كود إضافي لتحميل الملف مباشرة إذا كنتِ تستخدمين Google Colab
    from google.colab import files
    files.download(file_name)

except Exception as e:
    print(f"❌ حدث خطأ أثناء التشغيل: {e}")

--- جاري تشغيل الـ Agent... قد يستغرق ذلك وقتاً حسب عدد الروابط ---
--- Exreracting links from barcodes---
--- جاري تحليل 33 روابط ---
فشل الوصول للرابط: https://mim.gov.sa/youth-growth-program/ بسبب: HTTPSConnectionPool(host='mim.gov.sa', port=443): Max retries exceeded with url: /youth-growth-program/ (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7de54ea5a2d0>: Failed to establish a new connection: [Errno 101] Network is unreachable'))
فشل الوصول للرابط: https://sysdown.sawaeed.sa/companySurvey/contact/home بسبب: HTTPSConnectionPool(host='sysdown.sawaeed.sa', port=443): Max retries exceeded with url: /companySurvey/contact/home (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7de54ecabfe0>, 'Connection to sysdown.sawaeed.sa timed out. (connect timeout=10)'))
فشل الوصول للرابط: ksahr@carrier.com بسبب: Invalid URL 'ksahr@carrier.com': No scheme supplied. Perhaps you meant https://ksahr@carrier.com?
فشل الوصول للرابط: https:

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>